# Experiment Runner Notebook

This notebook automates the experiment lineup for Google Colab. It:
- Installs dependencies and prepares the repo
- Provides a runner to launch training runs (uses the repo's `core_ml/train/train.py`)
- Runs the experiment grid you requested (attention variants, windows, positional tests)
- After each run it performs an evaluation pass and appends results to a JSON file so progress is incremental.

Notes:
- The notebook expects the repository to be available in the Colab VM (e.g., mounted from Drive). Update `REPO_DIR` if needed.
- Heavy training should be run on an appropriate GPU runtime (Colab Pro recommended).

In [ ]:
# Environment & repo setup: clone repo and set WandB defaults
import os, sys, subprocess
import wandb
GITHUB_REPO = os.getenv('GITHUB_REPO', 'https://github.com/your-username/SAiDL-Summer-Assignment-2026.git')
REPO_DIR = os.getenv('REPO_DIR', '/content/SAiDL-Summer-Assignment-2026')
# WandB defaults (override with env vars in Colab):
os.environ.setdefault('WANDB_PROJECT', os.getenv('WANDB_PROJECT', 'SAiDL-CORE ML'))
os.environ.setdefault('WANDB_ENTITY', os.getenv('WANDB_ENTITY', 'VvS-2403'))

try:
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {GITHUB_REPO} to {REPO_DIR} ...")
        subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
    elif not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"Warning: {REPO_DIR} exists but is not a git repository.")
    else:
        print(f"Using existing repo at {REPO_DIR}")
except Exception as e:
    print("Repo clone/check failed:", e)

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    try:
        if os.path.exists("requirements.txt"):
            print("Installing requirements (may take a while)...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
        else:
            print("No requirements.txt found; install dependencies manually if needed.")
    except Exception as e:
        print("Requirements install failed (continuing):", e)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("Working directory:", os.getcwd())
else:
    raise FileNotFoundError(f"Unable to locate or clone repository at {REPO_DIR}")


In [ ]:
# WandB login (safe): prefer WANDB_API_KEY env or interactive login
import os, wandb
print("WandB version:", getattr(wandb, "__version__", "(unknown)"))
print("WandB project:", os.environ.get("WANDB_PROJECT"))
print("WandB entity:", os.environ.get("WANDB_ENTITY"))
try:
    api_key = os.environ.get("WANDB_API_KEY")
    if api_key:
        wandb.login(key=api_key)
    else:
        print("No WANDB_API_KEY found; running interactive wandb.login() if available in this environment.")
        try:
            wandb.login()
        except Exception as e:
            print("wandb.login() failed or deferred:", e)
except Exception as e:
    print("WandB import/login check failed (continuing):", e)


In [ ]:
import wandb
print("WandB version:", wandb.__version__)
os.environ["WANDB_PROJECT"] = "SAiDL-CORE ML"
print("Using WandB project:", os.environ["WANDB_PROJECT"])
wandb.login()


## Runner utilities


In [ ]:
from typing import Dict, Any, Optional
import subprocess, tempfile, os, sys, time
import math

# Safe defaults (fall back to sensible globals)
PYTHON_BIN = globals().get('PYTHON_BIN', sys.executable)
REPO_ROOT = globals().get('REPO_ROOT', os.getcwd())

# run_training: launch the repo training entrypoint in a subprocess
def run_training(overrides: Dict[str, Any], run_name: str, timeout: int = 60*60*6) -> Dict[str, Any]:
    # ensure experiment_name and output_dir are set
    ov = dict(overrides) if overrides is not None else {}
    ov['experiment_name'] = ov.get('experiment_name', run_name)
    out_dir = ov.get('output_dir') or os.path.join(REPO_ROOT, 'experiments', run_name)
    ov['output_dir'] = out_dir
    # build args for hydra: key=value pairs
    args = [PYTHON_BIN, '-m', 'core_ml.train.train']
    for k, v in ov.items():
        args.append(f'{k}={v}')
    try:
        proc = subprocess.Popen(args, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        stdout, stderr = proc.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        stdout, stderr = proc.communicate()
        return {'ok': False, 'timeout': True, 'stdout': stdout, 'stderr': stderr, 'output_dir': out_dir}
    return {'ok': proc.returncode == 0, 'returncode': proc.returncode, 'stdout': stdout, 'stderr': stderr, 'output_dir': out_dir}

# have been removed per request. Use the returned dicts from `run_training`
# and simple prints to inspect outputs instead.


: 

## Experiment grids and helper cells
The following cells contain the experiment grids; run them one-by-one.
Recommended batch-size mapping (tune to your GPU): {512:8, 1024:4, 2048:2, 4096:1}.


In [ ]:
# --- Baseline training (Colab) ---
# This cell runs the baseline `vanilla_mha` model with sequence length 1024.
# It is safe to run on Colab (will use the mounted repo) and will record results
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}

params = {
    'seq_len': 1024,
    'batch_size': bs_map[1024],
    'attention': 'vanilla_mha',
    'num_epochs': 1,          # set higher for full training
    'early_stop_steps': 2000, # optional short-circuit for faster runs
}

run_name = 'baseline_seq1024'
out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
os.makedirs(out_dir, exist_ok=True)
ov = {
    'dataset.seq_len': params['seq_len'],
    'dataset.batch_size': params['batch_size'],
    'attention.name': params['attention'],
    'training.num_epochs': params['num_epochs'],
    'training.early_stop_steps': params['early_stop_steps'],
    'experiment_name': run_name,
            'output_dir': out_dir,
}

print('Starting baseline run:', run_name)
train_res = run_training(ov, run_name, timeout=60*60*6)
print('Train finished. Success:', train_res['ok'])

eval_info = {'skipped': True}

print('Evaluation:', eval_info)


In [ ]:
# --- GQA grid: num_kv_heads in [2,1] across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
gqa_heads = [2, 1]
seq_grid = [512, 1024, 2048, 4096]

for heads in gqa_heads:
    for seq in seq_grid:
        params = {
            'seq_len': seq,
            'batch_size': bs_map[seq],
            'attention': 'gqa',
            'num_kv_heads': heads,
            'num_epochs': 1,
            'early_stop_steps': 2000,
        }
        run_name = f'gqa_heads{heads}_seq{seq}'
        out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': params['seq_len'],
            'dataset.batch_size': params['batch_size'],
            'attention.name': params['attention'],
            'attention.num_kv_heads': params['num_kv_heads'],
            'training.num_epochs': params['num_epochs'],
            'training.early_stop_steps': params['early_stop_steps'],
            'experiment_name': run_name,
            'output_dir': out_dir,
        }

        print('\n=== RUN:', run_name)
        train_res = run_training(ov, run_name, timeout=60*60*6)
        print('Train success:', train_res['ok'])
        try:
            eval_info = {'skipped': True}
        except Exception as e:
            eval_info = {'error': str(e)}
        print('Done', run_name, '->', eval_info)


In [ ]:
# --- Sparse attention experiments across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    params = {
        'seq_len': seq,
        'batch_size': bs_map[seq],
        'attention': 'sparse_attention',
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'sparse_seq{seq}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'experiment_name': run_name,
            'output_dir': out_dir,
    }

    print('\n=== RUN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
    try:
        eval_info = {'skipped': True}
    except Exception as e:
        eval_info = {'error': str(e)}
    print('Done', run_name, '->', eval_info)


In [ ]:
# --- Sparse attention experiments across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    params = {
        'seq_len': seq,
        'batch_size': bs_map[seq],
        'attention': 'sparse_attention',
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'sparse_seq{seq}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'experiment_name': run_name,
            'output_dir': out_dir,
    }

    print('\n=== RUN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
    try:
        eval_info = {'skipped': True}
    except Exception as e:
        eval_info = {'error': str(e)}
    print('Done', run_name, '->', eval_info)


In [ ]:
# --- Positional encoding experiments (RoPE, ALiBi, Relative) with Ltrain=512 and extrapolation evals ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
pos_variants = ['rope', 'alibi', 'relative']
Ltrain = 512
Ltests = [512, 1024, 2048]

for pos in pos_variants:
    params = {
        'seq_len': Ltrain,
        'batch_size': bs_map[Ltrain],
        'attention': 'vanilla_mha',
        'positional': pos,
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'pos_{pos}_train{Ltrain}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'positional.name': params['positional'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'experiment_name': run_name,
            'output_dir': out_dir,
    }

    print('\n=== TRAIN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train finished. Success:', train_res['ok'])

    for Ltest in Ltests:
        eval_params = params.copy()
        eval_params['seq_len'] = Ltest
        eval_ov = {
            'dataset.seq_len': eval_params['seq_len'],
            'dataset.batch_size': bs_map[Ltrain],
            'attention.name': eval_params['attention'],
            'positional.name': eval_params['positional'],
            'experiment_name': run_name,
            'output_dir': out_dir,
        }
        print(f'\n=== EVAL {pos} trained on {Ltrain} | test_seq={Ltest}')
eval_info = {'skipped': True}


In [ ]:
# --- Softmax (vanilla multi-head) across seq grid [512,1024,2048,4096] ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    params = {
        'seq_len': seq,
        'batch_size': bs_map[seq],
        'attention': 'vanilla_mha',
        'num_epochs': 1,
        'early_stop_steps': 2000,
    }
    run_name = f'softmax_seq{seq}'
    out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
    os.makedirs(out_dir, exist_ok=True)
    ov = {
        'dataset.seq_len': params['seq_len'],
        'dataset.batch_size': params['batch_size'],
        'attention.name': params['attention'],
        'training.num_epochs': params['num_epochs'],
        'training.early_stop_steps': params['early_stop_steps'],
        'experiment_name': run_name,
            'output_dir': out_dir,
    }

    print('\n=== RUN:', run_name)
    train_res = run_training(ov, run_name, timeout=60*60*6)
    print('Train success:', train_res['ok'])
    try:
        eval_info = {'skipped': True}
    except Exception as e:
        eval_info = {'error': str(e)}
    print('Done', run_name, '->', eval_info)


In [ ]:
# --- Sliding window experiments (window sizes seq/2, seq/4, seq/8) across seq grid ---
bs_map = {512: 8, 1024: 4, 2048: 2, 4096: 1}
seq_grid = [512, 1024, 2048, 4096]

for seq in seq_grid:
    for div in [2, 4, 8]:
        window = max(1, seq // div)
        params = {
            'seq_len': seq,
            'batch_size': bs_map[seq],
            'attention': 'sliding_window',
            'window_size': window,
            'num_epochs': 1,
            'early_stop_steps': 2000,
        }
        run_name = f'sliding_seq{seq}_w{window}'
        out_dir = os.path.join(REPO_ROOT, 'experiments', run_name)
        os.makedirs(out_dir, exist_ok=True)
        ov = {
            'dataset.seq_len': params['seq_len'],
            'dataset.batch_size': params['batch_size'],
            'attention.name': params['attention'],
            'attention.window_size': params['window_size'],
            'training.num_epochs': params['num_epochs'],
            'training.early_stop_steps': params['early_stop_steps'],
            'experiment_name': run_name,
            'output_dir': out_dir,
        }

        print('\n=== RUN:', run_name)
        train_res = run_training(ov, run_name, timeout=60*60*6)
        print('Train success:', train_res['ok'])
        try:
            eval_info = {'skipped': True}
        except Exception as e:
            eval_info = {'error': str(e)}
        print('Done', run_name, '->', eval_info)


## Complete experiment sequence
This notebook now contains the full requested lineup, one cell at a time:
1. Baseline with `seq_len=1024`
2. GQA grid with `num_kv_heads=2` and `1` across `[512, 1024, 2048, 4096]`
3. Sliding window experiments with window sizes `seq/2`, `seq/4`, `seq/8` across `[512, 1024, 2048, 4096]`
4. Softmax (vanilla multi-head) across `[512, 1024, 2048, 4096]`
5. Sparse attention across `[512, 1024, 2048, 4096]`
6. Positional encoding tests using RoPE, ALiBi, and Relative, plus extrapolation evaluation on `512`, `1024`, and `2048`.



In [ ]:
# Quick AST parse check for all code cells
errors = []
for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))
    try:
        ast.parse(src)
    except SyntaxError as e:
        errors.append((i, str(e)))
if errors:
    print("AST PARSE ERRORS:")
    for i, e in errors:
        print(i, e)
else:
    print("ALL CODE CELLS PARSE OK")
